# BG Parliament ASR — training progress

**Zipformer transducer** (icefall) trained from scratch on the Kaldi-aligned Bulgarian
parliament corpus. Purpose is twofold: a shippable BG ASR model **and** a WER benchmark
(mirroring the Finnish-parliament pipeline, arXiv 2203.14876).

This notebook is **re-runnable** — it parses the live training logs under
`~/icefall/egs/bg_parliament/ASR/zipformer/exp/log/` (across all power-cut restarts),
so just *Run All* to refresh. It does **not** touch the GPU or the running job.

| | |
|---|---|
| Backbone | Zipformer transducer (pruned), non-causal |
| Features | 80-dim fbank @ 16 kHz, no speed perturb (plain baseline) |
| Tokenizer | BPE-500 |
| Precision | fp16 (`--use-fp16 1`) |
| Schedule | Eden, 30 epochs, `--max-duration 350`, `--save-every-n 2000` |
| Hardware | RTX 5070 Ti (16 GB, sm_120) |

### Corpus (frozen split)

| Split | Cuts | Hours | Role |
|---|---|---|---|
| train | 419,915 | 649.2 | training |
| **dev** | 4,546 | 6.2 | tuning / model selection (touch freely) |
| **test** | 11,472 | 16.4 | final benchmark (touch **once**, at the end) |

## 1 · Parse the training logs

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

EXP     = Path.home() / "icefall/egs/bg_parliament/ASR/zipformer/exp"
LOG_DIR = EXP / "log"
DOCS    = Path.home() / "WORK/OmniVoice/docs"

logs = sorted(LOG_DIR.glob("log-train-*"))
print(f"{len(logs)} train log(s):")
for f in logs:
    print("  ", f.name)

tr, va = [], []
for f in logs:
    for line in f.read_text(errors="ignore").splitlines():
        if len(line) < 20 or line[4] != "-":          # cheap timestamp guard
            continue
        ts = line[:19]
        try:
            if ", validation: loss=" in line and "] Epoch " in line:
                ep = int(line.split("] Epoch ")[1].split(",")[0])
                vl = float(line.split("validation: loss=")[1].split(",")[0])
                va.append((ts, ep, vl))
            elif "tot_loss[loss=" in line and "] Epoch " in line and ", batch " in line:
                ep = int(line.split("] Epoch ")[1].split(",")[0])
                bt = int(line.split(", batch ")[1].split(",")[0])
                tl = float(line.split("tot_loss[loss=")[1].split(",")[0])
                lr = float(line.split(" lr: ")[1].split(",")[0]) if " lr: " in line else None
                gs = float(line.split(" grad_scale: ")[1].split()[0]) if " grad_scale: " in line else None
                tr.append((ts, ep, bt, tl, lr, gs))
        except (IndexError, ValueError):
            continue

df = pd.DataFrame(tr, columns=["ts", "epoch", "batch", "tot_loss", "lr", "grad_scale"])
dv = pd.DataFrame(va, columns=["ts", "epoch", "valid_loss"])
for d in (df, dv):
    d["ts"] = pd.to_datetime(d["ts"])
    d.sort_values("ts", inplace=True, ignore_index=True)

print(f"\nparsed {len(df):,} train points, {len(dv)} validation points")

2 train log(s):
   log-train-2026-08-02-21-34-52
   log-train-2026-08-03-02-38-18

parsed 4,407 train points, 92 validation points


## 2 · Run summary

In [2]:
t0   = df["ts"].iloc[0]
gaps = df["ts"].diff().dt.total_seconds().fillna(0)
idle = gaps[gaps > 600].sum()                     # gaps > 10 min = power cut / restart
span = (df["ts"].iloc[-1] - t0).total_seconds()
active_h = (span - idle) / 3600

print(f"run start          : {t0}")
print(f"latest log point   : {df['ts'].iloc[-1]}")
print(f"epochs seen         : {df['epoch'].min()}..{df['epoch'].max()}  (target 30)")
print(f"wall-clock span     : {span/3600:5.1f} h")
print(f"idle in gaps        : {idle/3600:5.1f} h  ({int((gaps>600).sum())} interruption(s))")
print(f"active training     : {active_h:5.1f} h")
print(f"latest train tot_loss: {df['tot_loss'].iloc[-1]:.4f}")
if len(dv):
    best = dv.loc[dv['valid_loss'].idxmin()]
    print(f"best dev loss       : {best['valid_loss']:.4f}  (epoch {int(best['epoch'])}, {best['ts']})")
    print(f"latest dev loss     : {dv['valid_loss'].iloc[-1]:.4f}")

run start          : 2026-08-02 21:35:39
latest log point   : 2026-08-03 16:15:09
epochs seen         : 1..30  (target 30)
wall-clock span     :  18.7 h
idle in gaps        :   4.0 h  (1 interruption(s))
active training     :  14.6 h
latest train tot_loss: 0.1565
best dev loss       : 0.1315  (epoch 30, 2026-08-03 15:58:52)
latest dev loss     : 0.1338


## 3 · Training loss

In [3]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df["ts"], df["tot_loss"], lw=0.7, color="#2b6cb0")
ax.set_yscale("log")
ax.set_ylabel("train tot_loss (log)")
ax.set_xlabel("wall-clock")
ax.set_title("BG Zipformer — training loss (dotted lines = epoch starts)")
ax.grid(True, which="both", alpha=0.3)
for ep in sorted(df["epoch"].unique()):
    ax.axvline(df[df["epoch"] == ep]["ts"].iloc[0], color="grey", ls=":", lw=0.4)
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(DOCS / "bg_asr_trainloss.png", dpi=110)
plt.show()

/tmp/ipykernel_965347/2575889431.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4 · Held-out dev loss (generalization proxy)

In [4]:
if len(dv):
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(dv["ts"], dv["valid_loss"], "-o", ms=4, color="#c05621", label="dev loss")
    bi = dv["valid_loss"].idxmin()
    ax.scatter(dv["ts"][bi], dv["valid_loss"][bi], s=90, facecolors="none",
               edgecolors="green", lw=2, label=f"best {dv['valid_loss'][bi]:.4f}")
    ax.set_ylabel("dev (validation) loss"); ax.set_xlabel("wall-clock")
    ax.set_title("BG Zipformer — held-out dev loss")
    ax.grid(True, alpha=0.3); ax.legend()
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(DOCS / "bg_asr_devloss.png", dpi=110)
    plt.show()
else:
    print("no validation points parsed yet")

/tmp/ipykernel_965347/2797595859.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5 · Optimizer health (lr & fp16 grad_scale)

`grad_scale` rising/stable = fp16 healthy; collapse toward 0 + NaN = danger.

In [5]:
fig, ax1 = plt.subplots(figsize=(11, 3.5))
ax1.plot(df["ts"], df["lr"], color="#6b46c1", lw=0.8)
ax1.set_ylabel("lr", color="#6b46c1"); ax1.set_yscale("log")
ax2 = ax1.twinx()
ax2.plot(df["ts"], df["grad_scale"], color="#dd6b20", lw=0.8, alpha=0.7)
ax2.set_ylabel("grad_scale", color="#dd6b20")
ax1.set_title("optimizer health — lr & fp16 grad_scale")
ax1.grid(True, alpha=0.3); fig.autofmt_xdate(); fig.tight_layout()
plt.show()

/tmp/ipykernel_965347/2860456178.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 · Checkpoint inventory

In [6]:
eps = sorted(int(p.stem.split("-")[1]) for p in EXP.glob("epoch-*.pt"))
cks = sorted(int(p.stem.split("-")[1]) for p in EXP.glob("checkpoint-*.pt"))
gib = lambda ps: sum(p.stat().st_size for p in ps) / 2**30

print(f"epoch checkpoints : {len(eps)} present  {eps}")
print(f"  size            : {gib(list(EXP.glob('epoch-*.pt'))):.1f} GiB  (all kept)")
print(f"mid-epoch ckpts   : {len(cks)} present  (rotating, keep_last_k=30)")
print(f"  range           : {cks[0]}..{cks[-1]}" if cks else "  (none)")
print(f"  size            : {gib(list(EXP.glob('checkpoint-*.pt'))):.1f} GiB")
print(f"exp dir *.pt total: {gib(list(EXP.glob('*.pt'))):.1f} GiB")

epoch checkpoints : 30 present  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
  size            : 29.3 GiB  (all kept)
mid-epoch ckpts   : 30 present  (rotating, keep_last_k=30)
  range           : 154000..212000
  size            : 29.3 GiB
exp dir *.pt total: 60.6 GiB


## 7 · WER results

> **Protocol.** Tune on **dev**; report **test** exactly once. The KenLM used for the
> test decode must be rebuilt on text that **excludes dev/test** transcripts, or the
> WER is optimistic. See `docs/BG_ASR_PLAN.md`.

Decode commands (run after epoch 30, GPU free):

```bash
cd ~/icefall/egs/bg_parliament/ASR
export PYTHONPATH=~/icefall
# dev sanity read (add a --eval-set dev switch to decode.py first):
python zipformer/decode.py --epoch 30 --avg 5 --exp-dir zipformer/exp \
    --bpe-model data/lang_bpe_500/bpe.model --max-duration 600 \
    --decoding-method greedy_search
# final benchmark on test:
python zipformer/decode.py --epoch 30 --avg 5 --exp-dir zipformer/exp \
    --bpe-model data/lang_bpe_500/bpe.model --max-duration 600 \
    --decoding-method modified_beam_search
```

**Dev ladder (2026-08-03, epoch 30 / avg 5, tuned on dev):**

| Date | Set | Method | LM | WER % |
|---|---|---|---|---|
| 2026-08-03 | dev | greedy_search | — | 5.29 |
| 2026-08-03 | dev | modified_beam_search (beam 4) | — | 5.19 |
| 2026-08-03 | dev | fast_beam_search_nbest_LG | word 3-gram @ scale 0.2 | **5.14** |
| _pending_ | **test** | greedy / beam / +LG@0.2 | —/—/3-gram | _TBD (decode once)_ |

The LG scale was picked from a dev sweep (0.01→0.3, clean U-min at 0.2). The n-gram gain
over beam is marginal (5.19→5.14, ~1% rel) — the 649h transducer learned the language so
well that an external n-gram adds little. KenLM built leakage-free (`build_kenlm_bg.sh`,
dev/test excluded); LG compiled by `compile_lg_bg.sh`.

A small **dev↔test gap** is itself evidence of healthy in-domain generalization;
a large gap means the tuning leaned too hard on dev.